# 05 — Anomaly scores, subtype complementarity and learned fusion

Evaluate classifier novelty, Isolation Forest, GP residuals and amplitude.
Known-class labels train the classifier; 99x labels define the evaluation target.
Study aggregate performance and subtype differences before testing score fusion.
The offline OOF fusion experiment is distinct from feedback-driven ranking in AL.


## 0. Configuration


In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'config.py').exists() and (p/'src').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import config
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from src.models import classification
from src.models import anomalies as an
RS = config.RANDOM_STATE


## 1. Calibrated E2-GP and evaluation labels


In [ ]:
tr = classification.features_e2gp('train')
X_tr, COLS = classification.e2gp_matrix(tr); y_tr = tr['target'].to_numpy()
RETRAIN_MODEL = False
if RETRAIN_MODEL:
    clf = classification.e2gp_al(y=y_tr, random_state=RS).fit(X_tr, y_tr)
else:
    clf, saved_columns, model_contract = classification.load_e2gp_artifact()
    if list(COLS) != list(saved_columns):
        raise ValueError('Current features do not match the saved model contract')
    print('Loaded the verified canonical model; classifier fitting was not repeated.')
meta_glob = pd.read_csv(config.TEST_METADATA_CSV, usecols=['object_id', 'true_target'])
print(f'train {X_tr.shape} · classifier E2-GP calibrated listo')


## 2. Four anomaly descriptors on batch 4


In [ ]:
sc4, y4, tt4 = an.batch_descriptors(4, clf, COLS, meta_glob)
print(f'batch 4: n={len(y4)} · anomalies 99x={int(y4.sum())} ({100*y4.mean():.2f}%)')
print('subtipos:', {int(s): int((tt4 == s).sum()) for s in [991, 992, 993, 994]})


## 3. Descriptor ranking quality


In [ ]:
tabla4 = an.evaluate_scores(sc4, y4, ks=(50, 126, 300))
tabla4.to_csv(config.DIR_PROCESSED / 'anomalias_descriptores_batch4.csv', index=False)
display(tabla4)


### 3b. Consistency across batches 1, 4, 5 and 6


In [ ]:
BATCHES = [1, 4, 5, 6]
data = {4: (sc4, y4, tt4)}
records_list = []
for b in BATCHES:
    if b not in data:
        data[b] = an.batch_descriptors(b, clf, COLS, meta_glob)
    sc, yb, _ = data[b]
    fila = {'batch': b, 'régimen': 'DDF' if b == 1 else 'WFD', 'anom': int(yb.sum())}
    fila.update({k: round(roc_auc_score(yb, sc[k]), 3) for k in sc})
    records_list.append(fila)
cons = pd.DataFrame(records_list)
cons.loc[len(cons)] = ['media', '', ''] + [round(cons[k].mean(), 3) for k in sc4]
cons.to_csv(config.DIR_PROCESSED / 'anomalias_consistencia_batches.csv', index=False)
display(cons)


descs = list(sc4); xb = np.arange(len(descs)); w = 0.2
fig, ax = plt.subplots(figsize=(7.6, 3.6))
for i, b in enumerate(BATCHES):
    sc, yb, _ = data[b]
    vals = [roc_auc_score(yb, sc[d]) for d in descs]
    ax.bar(xb + (i - 1.5) * w, vals, w, label=f'batch {b}' + (' (DDF)' if b == 1 else ''))
ax.axhline(0.5, ls=':', color='gray', lw=1)
ax.set_xticks(xb); ax.set_xticklabels(descs, rotation=15, ha='right')
ax.set_ylabel('ROC-AUC'); ax.set_ylim(0.35, 0.85)
ax.set_title('Descriptor consistency across batches (1 = DDF, 4/5/6 = WFD)')
ax.legend(fontsize=8, ncol=4); fig.tight_layout()
fig.savefig(config.FIGURE_DIR / 'anomalias_consistencia_batches.png', dpi=150); plt.show()


## 4. Complementarity by 99x subtype in the combined pool

Use the full processed pool for subtype diagnostics. No individual descriptor
is expected to dominate every astrophysical population.


In [ ]:
sc99, y99, tt99 = an.batch_descriptors(99, clf, COLS, meta_glob)
known_objects = tt99 < 990
records_list = []
for sub in [991, 992, 993, 994]:
    m = known_objects | (tt99 == sub)
    ysub = (tt99[m] == sub).astype(int)
    fila = {'anomalia': sub, 'n': int((tt99 == sub).sum())}
    fila.update({k: round(roc_auc_score(ysub, s[m]), 3) for k, s in sc99.items()})
    records_list.append(fila)
comp99 = pd.DataFrame(records_list)
comp99.to_csv(config.DIR_PROCESSED / 'anomalias_complementariedad_pool.csv', index=False)
display(comp99)

M = comp99[list(sc99)].to_numpy(float)
fig, ax = plt.subplots(figsize=(7, 3.2))
im = ax.imshow(M, cmap='viridis', vmin=0.4, vmax=0.9, aspect='auto')
ax.set_xticks(range(len(sc99))); ax.set_xticklabels(list(sc99), rotation=20, ha='right')
ax.set_yticks(range(4))
ax.set_yticklabels([f'{s} (n={n})' for s, n in zip(comp99['anomalia'], comp99['n'])])
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.text(j, i, f'{M[i,j]:.2f}', ha='center', va='center',
                color='w' if M[i, j] < 0.7 else 'k', fontsize=9)
fig.colorbar(im, label='ROC-AUC')
ax.set_title('Complementarity by anomaly subtype (eleven-batch pool)')
fig.tight_layout(); fig.savefig(config.FIGURE_DIR / 'anomalias_complementariedad_pool.png', dpi=150); plt.show()


## 5. Compare fusion rules

Compare classifier novelty, equal-weight ranks, a fixed heuristic and a learned
logistic re-ranker evaluated out of fold. ROC-AUC summarizes global ordering;
precision at K describes the part of the ranking relevant to a small query budget.
This experiment uses evaluation labels offline and is not the deployed AL policy.


In [ ]:
pesos_oraculo = {'novedad_1-conf': 0.7, 'IsolationForest': 0.15, 'residuos_GP': 0.1, 'cambios_mag': 0.05}
def _R(sc): return np.column_stack([an.rank01(sc[k]) for k in sc])
def _fila(item_name, y, s):
    p = lambda k: float(np.asarray(y)[np.argsort(s)[::-1][:k]].mean())
    n = int(np.asarray(y).sum())
    return {'método': item_name, 'ROC_AUC': round(roc_auc_score(y, s), 4),
            'P@100': round(p(100), 4), 'P@300': round(p(300), 4), 'P@N_anom': round(p(n), 4)}

cv = StratifiedKFold(5, shuffle=True, random_state=RS)
R99 = _R(sc99)
oof99 = cross_val_predict(LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced'),
                          R99, y99, cv=cv, method='predict_proba')[:, 1]
comp = pd.DataFrame([
    _fila('novelty only (best single descriptor)', y99, sc99['novedad_1-conf']),
    _fila('Naive fusion (equal weights)', y99, an.fuse_scores(sc99)),
    _fila('Heuristic fusion (fixed manual weights)', y99, an.fuse_scores(sc99, score_weights=pesos_oraculo)),
    _fila('OOF reranker (learned weights)', y99, oof99),
]).sort_values('ROC_AUC', ascending=False).reset_index(drop=True)
comp.to_csv(config.DIR_PROCESSED / 'anomalias_fusion_reranker.csv', index=False)
display(comp)

lr_full = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced').fit(R99, y99)
print('Learned logistic coefficients in the pool:',
      pd.Series(lr_full.coef_[0], index=list(sc99)).round(3).to_dict())

fig, ax = plt.subplots(figsize=(6.2, 5.2))
for item_name, s, c, ls in [('novelty (1−conf)', sc99['novedad_1-conf'], 'C0', '-'),
                          ('Heuristic fusion (fixed manual weights)', an.fuse_scores(sc99, score_weights=pesos_oraculo), 'C7', '--'),
                          ('OOF reranker (learned)', oof99, 'C3', '-')]:
    fpr, tpr, _ = roc_curve(y99, s)
    ax.plot(fpr, tpr, ls, color=c, label=f'{item_name} · AUC={roc_auc_score(y99, s):.3f}')
ax.plot([0, 1], [0, 1], ':', color='gray', lw=1)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('Descriptor fusion · eleven-batch pool')
ax.legend(loc='lower right', fontsize=8); ax.grid(alpha=.3); fig.tight_layout()
fig.savefig(config.FIGURE_DIR / 'anomalias_fusion_roc_pool.png', dpi=150); plt.show()


## Interpretation

The descriptors are complementary by subtype. Learned fusion can improve the
static ranking, but its sequential value must be evaluated under the same query
budget and feedback restrictions as the classification channel.
